# Lesson 07 - Planning Design Pattern

This notebook demonstrates the **Planning Design Pattern** for AI agents using the Microsoft Agent Framework.
You will learn how to break complex travel requests into structured subtasks, assign them to specialist agents,
and execute the resulting plan — all using structured output powered by Pydantic models.


## Diegimas


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Užduoties suskaidymas

Užduoties suskaidymas yra planavimo dizaino šablono pagrindas. Vietoj to, kad vienas agentas
tvarkytų sudėtingą užklausą nuo pradžios iki galo, mes skaidome problemą į mažesnes, aiškiai apibrėžtas **použduotis**.
Kiekviena požyda priskiriama specializuotam agentui (pvz., skrydžiai, viešbučiai, veiklos), su aiškiomis
prioriteto ir priklausomybės tvarkomis.

Šis požiūris suteikia kelias naudas:
- **Aiškumas**: kiekviena požyda turi vieną atsakomybę.
- **Lygiagretumas**: nepriklausomos požydžiai gali vykti vienu metu.
- **Patikimumas**: klaidos izoliuojamos prie atskirų požyčių.
- **Biudžeto sekimas**: sąnaudos vertinamos pagal požydį ir sujungiamos.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Planavimo agento kūrimas su struktūruotu išvestimi

Planavimo agentas veikia kaip **priekinės registratūros koordinatorių**. Gavęs aukšto lygio kelionės užklausą,
jis sukuria struktūruotą `TravelPlan` — suskaidydamas užklausą į smulkesnes užduotis, nustatydamas prioritetus
ir identifikuodamas priklausomybes, kad konsjeržas ar vykdymo sluoksnis galėtų atlikti darbą.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Planu įgyvendinimas naudojant specializuotus įrankius

Kai registratūros darbuotojas parengia struktūrizuotą planą, jį įgyvendina **konsjeržo agentas**.
Kiekvienas specializuotas įrankis tvarko vieną užduočių kategoriją (skrydžiai, viešbučiai, veiklos). Konsjeržas
iteruoja per plano posavybes priklausomybės tvarka ir perduoda kiekvieną užduotį
atitinkamam įrankiui.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Santrauka

Šioje pamokoje išmokote **Planavimo dizaino modelį** AI agentams:

1. **Užduočių suskaidymas** — registratūros planavimo agentas suskaido sudėtingą kelionės užklausą į
   struktūrizuotas po-užduotis naudojant Pydantic modelius, priskirdamas kiekvieną specialistų agentui su prioritetais
   ir priklausomybėmis.
2. **Struktūrizuotas išvestis** — perduodamas `response_format`, agentas grąžina patvirtintą
   `TravelPlan` objektą vietoje laisvo teksto, todėl tolesnis apdorojimas yra patikimas.
3. **Plano vykdymas** — konsjeržo agentas iteratyviai atlieka po-užduotis naudodamas specialistų įrankius
   (`book_flight`, `reserve_hotel`, `book_activity`) plano įgyvendinimui ir rezultatų pranešimui.

Šis modelis atskiria *ką daryti* (planavimas) nuo *kaip tai atlikti* (vykdymas), todėl agentai
yra moduliniai, lengviau testuojami ir juos paprasčiau plėsti.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
